# 🍅 Tomato Crop Growth Progression Model
### LSTM-based Multi-Task Temporal Progression Predictor

**Stages:** `seedling → early_vegetative → flowering_initiation → flowering → unripe → ripe`

**Predictions per row:**
1. Current stage classification
2. Next stage classification
3. Hours to next stage (regression)
4. Probability of transition within 24h (binary)
5. Probability of transition within 48h (binary)

---

## SECTION 0 — Run ID & Path Setup

In [1]:
# ─────────────────────────────────────────────────────────────
# SECTION 0: Run ID & Centralised Path Setup
# ─────────────────────────────────────────────────────────────

import os
import logging
from datetime import datetime

# ── Generate a unique run ID for this execution ───────────────
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run ID : {RUN_ID}')

# ── Root project path ─────────────────────────────────────────
PROJECT_ROOT = r'E:\AgriTwin-GH'

# ── Dataset path ──────────────────────────────────────────────
DATA_PATH = os.path.join(
    PROJECT_ROOT, 'data', 'processed', 'Growth Progression',
    'tomato_growth_progression_synthetic_hourly.csv'
)

# ── Model save path ───────────────────────────────────────────
MODEL_DIR  = os.path.join(PROJECT_ROOT, 'src', 'agritwin_gh', 'models')
MODEL_PATH = os.path.join(MODEL_DIR, f'growth_stage_progression_{RUN_ID}.keras')

# ── Artifacts sub-structure ───────────────────────────────────
ARTIFACTS_DIR = os.path.join(MODEL_DIR, 'artifacts', f'growth_stage_progression_{RUN_ID}')
PLOTS_DIR     = ARTIFACTS_DIR
METRICS_DIR   = ARTIFACTS_DIR
LOGS_DIR      = ARTIFACTS_DIR
REPORTS_DIR   = ARTIFACTS_DIR

# ── Checkpoint path (used during training) ────────────────────
CKPT_PATH = os.path.join(ARTIFACTS_DIR, f'best_model_{RUN_ID}.keras')

# ── Create all directories ────────────────────────────────────
for d in [MODEL_DIR, ARTIFACTS_DIR, PLOTS_DIR, METRICS_DIR, LOGS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Run log setup ─────────────────────────────────────────────
LOG_FILE = os.path.join(LOGS_DIR, f'run_{RUN_ID}.log')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger('growth_progression')
logger.info(f'Run started | run_id={RUN_ID}')

print(f'Dataset         : {DATA_PATH}')
print(f'Model save path : {MODEL_PATH}')
print(f'Artifacts dir   : {ARTIFACTS_DIR}')
print(f'Log file        : {LOG_FILE}')
print('\n✅ Paths configured.')


2026-03-10 19:20:38,407 | INFO | Run started | run_id=20260310_192038


Run ID : 20260310_192038
Dataset         : E:\AgriTwin-GH\data\processed\Growth Progression\tomato_growth_progression_synthetic_hourly.csv
Model save path : E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_progression_20260310_192038.keras
Artifacts dir   : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038
Log file        : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\run_20260310_192038.log

✅ Paths configured.


## SECTION 1 — Setup: Libraries, Seeds, Version Info

In [2]:
# ─────────────────────────────────────────────────────────────
# SECTION 1: Setup
# ─────────────────────────────────────────────────────────────

import sys
import json
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')        # non-interactive backend for reliable save
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from collections import Counter
from pathlib import Path

# Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, mean_absolute_error,
    mean_squared_error, roc_auc_score, ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
)

# Optional SHAP
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not available — skipping SHAP plots.')

# ── Reproducibility ──────────────────────────────────────────
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# ── Helper: save a figure to the plots directory ─────────────
def save_fig(fig, name: str, subdir: str = '') -> str:
    """Save *fig* to PLOTS_DIR (optionally a subdir) and return path."""
    target_dir = os.path.join(PLOTS_DIR, subdir) if subdir else PLOTS_DIR
    os.makedirs(target_dir, exist_ok=True)
    path = os.path.join(target_dir, f'{name}.png')
    fig.savefig(path, bbox_inches='tight', dpi=150)
    return path

# ── Version info ─────────────────────────────────────────────
print(f'Python  : {sys.version}')
print(f'TF      : {tf.__version__}')
print(f'Keras   : {keras.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'GPU     : {tf.config.list_physical_devices("GPU")}')
logger.info(f'TF={tf.__version__}  GPU={tf.config.list_physical_devices("GPU")}')
print('\n✅ Setup complete.')


2026-03-10 19:23:54,097 | INFO | TF=2.20.0  GPU=[]


SHAP not available — skipping SHAP plots.
Python  : 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
TF      : 2.20.0
Keras   : 3.13.2
NumPy   : 2.4.2
Pandas  : 3.0.1
GPU     : []

✅ Setup complete.


## SECTION 2 — Data Loading

In [3]:
# ─────────────────────────────────────────────────────────────
# SECTION 2: Data Loading  (local file — no Colab upload needed)
# ─────────────────────────────────────────────────────────────

print(f'Loading dataset from:\n  {DATA_PATH}')
assert os.path.exists(DATA_PATH), f'Dataset not found: {DATA_PATH}'
raw_df = pd.read_csv(DATA_PATH)
logger.info(f'Dataset loaded | shape={raw_df.shape} | path={DATA_PATH}')

# ── Quick overview ────────────────────────────────────────────
print(f'\nShape   : {raw_df.shape}')
print(f'Columns : {list(raw_df.columns)}')
print('\n── Head ──')
display(raw_df.head(5))
print('\n── Info ──')
raw_df.info()
print('\n── Missing values ──')
print(raw_df.isnull().sum())

# ── Save dataset summary to artifacts ────────────────────────
dataset_summary = {
    'dataset_path'  : DATA_PATH,
    'n_rows'        : int(raw_df.shape[0]),
    'n_cols'        : int(raw_df.shape[1]),
    'columns'       : list(raw_df.columns),
    'dtypes'        : {c: str(t) for c, t in raw_df.dtypes.items()},
    'missing_values': raw_df.isnull().sum().to_dict(),
    'memory_usage_mb': round(raw_df.memory_usage(deep=True).sum() / 1e6, 3),
}
with open(os.path.join(METRICS_DIR, 'dataset_summary.json'), 'w') as f:
    json.dump(dataset_summary, f, indent=2, default=str)
print(f'\n✅ Dataset summary saved.')


Loading dataset from:
  E:\AgriTwin-GH\data\processed\Growth Progression\tomato_growth_progression_synthetic_hourly.csv


2026-03-10 19:24:22,629 | INFO | Dataset loaded | shape=(10848, 37) | path=E:\AgriTwin-GH\data\processed\Growth Progression\tomato_growth_progression_synthetic_hourly.csv



Shape   : (10848, 37)
Columns : ['timestamp', 'cycle_id', 'cycle_label', 'season_window', 'real_or_synthetic_flag', 'year', 'month', 'day_of_year', 'week_of_year', 'hour', 'season_label', 'days_from_cycle_start', 'stage_name', 'stage_index', 'hours_in_current_stage', 'days_in_current_stage', 'stage_duration_hours', 'stage_duration_days', 'stage_progress_pct', 'total_cycle_progress_pct', 'estimated_days_to_next_stage', 'estimated_hours_to_next_stage', 'is_stage_transition', 'indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_CO2', 'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy', 'light_period_flag', 'cumulative_gdd_like_index']

── Head ──


,timestamp,cycle_id,cycle_label,season_window,real_or_synthetic_flag,year,month,day_of_year,week_of_year,hour,season_label,days_from_cycle_start,stage_name,stage_index,hours_in_current_stage,days_in_current_stage,stage_duration_hours,stage_duration_days,stage_progress_pct,total_cycle_progress_pct,estimated_days_to_next_stage,estimated_hours_to_next_stage,is_stage_transition,indoor_temp,indoor_humidity,indoor_air_velocity,indoor_CO2,solarradiation,day_night_flag,vpd,dew_point,leaf_wetness_proxy,temperature_rolling_mean_24h,humidity_rolling_mean_24h,vpd_proxy,light_period_flag,cumulative_gdd_like_index
0,2024-07-01 00:00:00,1,kharif_2024,Kharif (Jun–Jul 2024),synthetic,2024,7,183,27,0,southwest_monsoon,0.0000,seedling,0,0.0,0.0000,288,12,0.000,0.000,12.0000,288.0,False,29.002223,75.929629,2.41,440.0,0.0,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0,0.7918
1,2024-07-01 01:00:00,1,kharif_2024,Kharif (Jun–Jul 2024),synthetic,2024,7,183,27,1,southwest_monsoon,0.0417,seedling,0,1.0,0.0417,288,12,0.347,0.037,11.9583,287.0,False,27.801924,70.830127,2.41,400.0,0.0,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,0,1.5335
2,2024-07-01 02:00:00,1,kharif_2024,Kharif (Jun–Jul 2024),synthetic,2024,7,183,27,2,southwest_monsoon,0.0833,seedling,0,2.0,0.0833,288,12,0.694,0.074,11.9167,286.0,False,28.278680,70.035534,2.41,400.0,0.0,1.0,1.151145,22.285786,0.0,28.3609,72.2651,1.1511,0,2.2951
3,2024-07-01 03:00:00,1,kharif_2024,Kharif (Jun–Jul 2024),synthetic,2024,7,183,27,3,southwest_monsoon,0.1250,seedling,0,3.0,0.1250,288,12,1.042,0.111,11.8750,285.0,False,28.900000,69.000000,2.41,400.0,0.0,1.0,1.234602,22.700000,0.0,28.4957,71.4488,1.2346,0,3.0826
4,2024-07-01 04:00:00,1,kharif_2024,Kharif (Jun–Jul 2024),synthetic,2024,7,183,27,4,southwest_monsoon,0.1667,seedling,0,4.0,0.1667,288,12,1.389,0.147,11.8333,284.0,False,29.623543,67.794095,2.41,400.0,0.0,1.0,1.337286,23.182362,0.0,28.7213,70.7179,1.3373,0,3.9003



── Info ──
<class 'pandas.DataFrame'>
RangeIndex: 10848 entries, 0 to 10847
Data columns (total 37 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   timestamp                      10848 non-null  str    
 1   cycle_id                       10848 non-null  int64  
 2   cycle_label                    10848 non-null  str    
 3   season_window                  10848 non-null  str    
 4   real_or_synthetic_flag         10848 non-null  str    
 5   year                           10848 non-null  int64  
 6   month                          10848 non-null  int64  
 7   day_of_year                    10848 non-null  int64  
 8   week_of_year                   10848 non-null  int64  
 9   hour                           10848 non-null  int64  
 10  season_label                   10848 non-null  str    
 11  days_from_cycle_start          10848 non-null  float64
 12  stage_name                     10848 non-null

## SECTION 3 — Data Standardization

In [5]:
# ─────────────────────────────────────────────────────────────
# SECTION 3: Data Standardization
# ─────────────────────────────────────────────────────────────

# ── 3.1  Canonical stage mapping ─────────────────────────────
STAGE_ORDER = [
    'seedling',
    'early_vegetative',
    'flowering_initiation',
    'flowering',
    'unripe',
    'ripe',
]
STAGE_TO_INT = {s: i for i, s in enumerate(STAGE_ORDER)}
INT_TO_STAGE = {i: s for s, i in STAGE_TO_INT.items()}
N_STAGES = len(STAGE_ORDER)
print('Stage mapping:', STAGE_TO_INT)

# ── 3.2  Column name aliases ──────────────────────────────────
ALIAS_MAP = {
    'time': 'timestamp', 'datetime': 'timestamp', 'date_time': 'timestamp',
    'date': 'timestamp', 'recorded_at': 'timestamp', 'ts': 'timestamp',
    'cycle': 'cycle_id', 'batch': 'cycle_id', 'batch_id': 'cycle_id',
    'crop_cycle': 'cycle_id', 'run_id': 'cycle_id',
    'growth_stage': 'stage', 'crop_stage': 'stage', 'phenological_stage': 'stage',
    'label': 'stage', 'stage_label': 'stage', 'phase': 'stage', 'stage_name': 'stage',
    'air_temperature': 'temperature', 'air_temp': 'temperature',
    'temp': 'temperature', 'temp_c': 'temperature', 'temperature_c': 'temperature',
    'relative_humidity': 'humidity', 'rh': 'humidity', 'rh_pct': 'humidity',
    'air_humidity': 'humidity',
    'light_intensity': 'light', 'par': 'light', 'ppfd': 'light',
    'dli': 'light', 'solar_radiation': 'light', 'irradiance': 'light',
    'soil_moisture': 'soil_moisture', 'vwc': 'soil_moisture',
    'soil_water': 'soil_moisture', 'moisture': 'soil_moisture',
    'co2_ppm': 'co2', 'carbon_dioxide': 'co2', 'co2_concentration': 'co2',
}


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    df.rename(columns={k: v for k, v in ALIAS_MAP.items() if k in df.columns},
              inplace=True)
    return df


def detect_required_columns(df: pd.DataFrame) -> dict:
    required = ['timestamp', 'cycle_id', 'stage']
    detected = {}
    missing = []
    for col in required:
        if col in df.columns:
            detected[col] = col
        else:
            missing.append(col)
    if missing:
        raise ValueError(
            f'Missing required columns after alias mapping: {missing}\n'
            f'Available columns: {list(df.columns)}'
        )
    optional = ['temperature', 'humidity', 'light', 'soil_moisture', 'co2']
    for col in optional:
        if col in df.columns:
            detected[col] = col
    print('\n── Detected columns ──')
    for k, v in detected.items():
        print(f'  {k:20s} → {v}')
    extra = [c for c in df.columns if c not in list(detected.values()) + required + optional]
    print(f'  Extra / unknown columns: {extra}')
    return detected


# ── 3.3  Stage label normalisation ───────────────────────────
STAGE_SPELLING_MAP = {
    'Seedling': 'seedling', 'SEEDLING': 'seedling',
    'early vegetative': 'early_vegetative',
    'earlyvegetative': 'early_vegetative',
    'early_veg': 'early_vegetative',
    'vegetative': 'early_vegetative',
    'flowering initiation': 'flowering_initiation',
    'floweringinitiation': 'flowering_initiation',
    'flower_initiation': 'flowering_initiation',
    'Flowering': 'flowering', 'FLOWERING': 'flowering',
    'Unripe': 'unripe', 'UNRIPE': 'unripe', 'green_fruit': 'unripe',
    'Ripe': 'ripe', 'RIPE': 'ripe', 'mature': 'ripe', 'harvest': 'ripe',
}


def normalise_stage_labels(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', '_', regex=True)
    )
    cleaned = cleaned.replace({k.lower().replace(' ', '_'): v
                                for k, v in STAGE_SPELLING_MAP.items()})
    return cleaned


# ── 3.4  Apply standardisation ────────────────────────────────
df = standardize_columns(raw_df)
detected_cols = detect_required_columns(df)

df['stage'] = normalise_stage_labels(df['stage'])
df['stage_int'] = df['stage'].map(STAGE_TO_INT)
invalid_mask = df['stage_int'].isna()
if invalid_mask.sum() > 0:
    print(f'\n⚠️  Removing {invalid_mask.sum()} rows with unrecognised stage labels:')
    print(df.loc[invalid_mask, 'stage'].value_counts())
    df = df[~invalid_mask].copy()
df['stage_int'] = df['stage_int'].astype(int)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df.sort_values(['cycle_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nData after standardisation: {df.shape}')
display(df.head(3))
logger.info(f'Standardisation complete | shape={df.shape}')


Stage mapping: {'seedling': 0, 'early_vegetative': 1, 'flowering_initiation': 2, 'flowering': 3, 'unripe': 4, 'ripe': 5}

── Detected columns ──
  timestamp            → timestamp
  cycle_id             → cycle_id
  stage                → stage
  Extra / unknown columns: ['cycle_label', 'season_window', 'real_or_synthetic_flag', 'year', 'month', 'day_of_year', 'week_of_year', 'hour', 'season_label', 'days_from_cycle_start', 'stage_index', 'hours_in_current_stage', 'days_in_current_stage', 'stage_duration_hours', 'stage_duration_days', 'stage_progress_pct', 'total_cycle_progress_pct', 'estimated_days_to_next_stage', 'estimated_hours_to_next_stage', 'is_stage_transition', 'indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_co2', 'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy', 'light_period_flag', 'cumulative_gdd_like_index']

Data after standardisation: (10848, 38)


,timestamp,cycle_id,cycle_label,season_window,real_or_synthetic_flag,year,month,day_of_year,week_of_year,hour,season_label,days_from_cycle_start,stage,stage_index,hours_in_current_stage,days_in_current_stage,stage_duration_hours,stage_duration_days,stage_progress_pct,total_cycle_progress_pct,estimated_days_to_next_stage,estimated_hours_to_next_stage,is_stage_transition,indoor_temp,indoor_humidity,indoor_air_velocity,indoor_co2,solarradiation,day_night_flag,vpd,dew_point,leaf_wetness_proxy,temperature_rolling_mean_24h,humidity_rolling_mean_24h,vpd_proxy,light_period_flag,cumulative_gdd_like_index,stage_int
0,2024-07-01 00:00:00,1,kharif_2024,Kharif (Jun–Jul 2024),synthetic,2024,7,183,27,0,southwest_monsoon,0.0000,seedling,0,0.0,0.0000,288,12,0.000,0.000,12.0000,288.0,False,29.002223,75.929629,2.41,440.0,0.0,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0,0.7918,0
1,2024-07-01 01:00:00,1,kharif_2024,Kharif (Jun–Jul 2024),synthetic,2024,7,183,27,1,southwest_monsoon,0.0417,seedling,0,1.0,0.0417,288,12,0.347,0.037,11.9583,287.0,False,27.801924,70.830127,2.41,400.0,0.0,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,0,1.5335,0
2,2024-07-01 02:00:00,1,kharif_2024,Kharif (Jun–Jul 2024),synthetic,2024,7,183,27,2,southwest_monsoon,0.0833,seedling,0,2.0,0.0833,288,12,0.694,0.074,11.9167,286.0,False,28.278680,70.035534,2.41,400.0,0.0,1.0,1.151145,22.285786,0.0,28.3609,72.2651,1.1511,0,2.2951,0


2026-03-10 19:26:20,527 | INFO | Standardisation complete | shape=(10848, 38)


## SECTION 4 — Cycle Integrity Checks

In [6]:
# ─────────────────────────────────────────────────────────────
# SECTION 4: Cycle Integrity Checks
# ─────────────────────────────────────────────────────────────

MIN_ROWS_PER_CYCLE = 10
REQUIRE_START_STAGE = 0
REQUIRE_END_STAGE   = 5


def check_cycle(grp: pd.DataFrame) -> dict:
    grp = grp.sort_values('timestamp')
    cid = grp['cycle_id'].iloc[0]
    stages_present = sorted(grp['stage_int'].unique().tolist())
    n_rows = len(grp)
    start_time = grp['timestamp'].min()
    end_time   = grp['timestamp'].max()
    duration_hours = (end_time - start_time).total_seconds() / 3600

    issues = []
    if not grp['timestamp'].is_monotonic_increasing:
        issues.append('non_monotonic_timestamps')
    n_dups = grp['timestamp'].duplicated().sum()
    if n_dups > 0:
        issues.append(f'{n_dups}_duplicate_timestamps')
    if n_rows < MIN_ROWS_PER_CYCLE:
        issues.append(f'only_{n_rows}_rows')
    stage_diffs = grp['stage_int'].diff().dropna()
    if (stage_diffs < 0).any():
        n_reversals = (stage_diffs < 0).sum()
        issues.append(f'{n_reversals}_stage_reversals')
    expected_stages = set(range(REQUIRE_START_STAGE, REQUIRE_END_STAGE + 1))
    missing_stages = expected_stages - set(stages_present)
    if missing_stages:
        missing_names = [INT_TO_STAGE[s] for s in sorted(missing_stages)]
        issues.append(f'missing_stages:{missing_names}')

    return {
        'cycle_id'      : cid,
        'n_rows'        : n_rows,
        'stages_present': stages_present,
        'start_time'    : start_time,
        'end_time'      : end_time,
        'duration_hours': round(duration_hours, 1),
        'issues'        : '; '.join(issues) if issues else 'OK',
    }


cycle_summary_records = [
    check_cycle(grp) for _, grp in df.groupby('cycle_id', sort=False)
]
cycle_summary = pd.DataFrame(cycle_summary_records)

print('── Cycle Summary ──')
display(cycle_summary)

# ── 4.2  Flag bad cycles ──────────────────────────────────────
bad_cycle_mask = cycle_summary['issues'] != 'OK'
bad_cycles  = cycle_summary.loc[bad_cycle_mask,  'cycle_id'].tolist()
good_cycles = cycle_summary.loc[~bad_cycle_mask, 'cycle_id'].tolist()

print(f'\nTotal cycles : {len(cycle_summary)}')
print(f'Good cycles  : {len(good_cycles)}')
print(f'Bad  cycles  : {len(bad_cycles)}  → {bad_cycles}')

# ── 4.3  Remove hard-error cycles & deduplicate timestamps ────
HARD_ERROR_KEYWORDS = ['stage_reversals', 'only_']


def is_hard_error(issue_str: str) -> bool:
    return any(kw in issue_str for kw in HARD_ERROR_KEYWORDS)


hard_bad = cycle_summary.loc[
    cycle_summary['issues'].apply(is_hard_error), 'cycle_id'
].tolist()

print(f'\nCycles removed (hard errors): {hard_bad}')
df = df[~df['cycle_id'].isin(hard_bad)].copy()
df = df.drop_duplicates(subset=['cycle_id', 'timestamp'], keep='last')
df.sort_values(['cycle_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nData after cycle cleaning: {df.shape}')
logger.info(f'Cycle cleaning done | good={len(good_cycles)} bad={len(bad_cycles)} hard_removed={len(hard_bad)}')

# ── Save cycle summary ────────────────────────────────────────
cycle_summary_path = os.path.join(METRICS_DIR, 'cycle_summary.csv')
cycle_summary.to_csv(cycle_summary_path, index=False)
print(f'Cycle summary saved → {cycle_summary_path}')


── Cycle Summary ──


,cycle_id,n_rows,stages_present,start_time,end_time,duration_hours,issues
0,1,2712,"[0, 1, 2, 3, 4, 5]",2024-07-01,2024-10-21 23:00:00,2711.0,OK
1,2,2616,"[0, 1, 2, 3, 4, 5]",2024-11-10,2025-02-26 23:00:00,2615.0,OK
2,3,2808,"[0, 1, 2, 3, 4, 5]",2025-03-05,2025-06-29 23:00:00,2807.0,OK
3,4,2712,"[0, 1, 2, 3, 4, 5]",2025-07-01,2025-10-21 23:00:00,2711.0,OK


2026-03-10 19:26:24,342 | INFO | Cycle cleaning done | good=4 bad=0 hard_removed=0



Total cycles : 4
Good cycles  : 4
Bad  cycles  : 0  → []

Cycles removed (hard errors): []

Data after cycle cleaning: (10848, 38)
Cycle summary saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\cycle_summary.csv


## SECTION 5 — Progression Target Construction

In [7]:
# ─────────────────────────────────────────────────────────────
# SECTION 5: Progression Target Construction
# ─────────────────────────────────────────────────────────────

def build_progression_targets(cycle_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each row in a single-cycle DataFrame, derive:
      current_stage, next_stage, hours_to_next_stage,
      transition_within_24h, transition_within_48h,
      stage_progress_fraction.
    """
    df_c = cycle_df.sort_values('timestamp').copy()
    n = len(df_c)

    next_stage_arr = np.full(n, np.nan)
    hours_to_next  = np.full(n, np.nan)
    trans_24h      = np.full(n, np.nan)
    trans_48h      = np.full(n, np.nan)
    stage_frac     = np.full(n, np.nan)

    stage_vals = df_c['stage_int'].values
    ts_vals    = df_c['timestamp'].values

    for i in range(n):
        cur_stage = stage_vals[i]
        if cur_stage == (N_STAGES - 1):
            continue
        for j in range(i + 1, n):
            if stage_vals[j] > cur_stage:
                dt_hours = (
                    (ts_vals[j] - ts_vals[i]).astype('timedelta64[s]').astype(float)
                    / 3600.0
                )
                next_stage_arr[i] = stage_vals[j]
                hours_to_next[i]  = dt_hours
                trans_24h[i]      = 1.0 if dt_hours <= 24 else 0.0
                trans_48h[i]      = 1.0 if dt_hours <= 48 else 0.0
                break

    for stage_idx in range(N_STAGES):
        mask = stage_vals == stage_idx
        if not mask.any():
            continue
        stage_ts = ts_vals[mask]
        t_start  = stage_ts.min()
        t_end    = stage_ts.max()
        total = (t_end - t_start).astype('timedelta64[s]').astype(float)
        if total > 0:
            elapsed = (stage_ts - t_start).astype('timedelta64[s]').astype(float)
            stage_frac[mask] = elapsed / total
        else:
            stage_frac[mask] = 0.0

    df_c['current_stage']          = stage_vals
    df_c['next_stage']             = next_stage_arr
    df_c['hours_to_next_stage']    = hours_to_next
    df_c['transition_within_24h']  = trans_24h
    df_c['transition_within_48h']  = trans_48h
    df_c['stage_progress_fraction'] = stage_frac
    return df_c


print('Building progression targets (may take a moment)...')
processed_chunks = [
    build_progression_targets(grp)
    for _, grp in df.groupby('cycle_id', sort=False)
]
df = pd.concat(processed_chunks, ignore_index=True)
df.sort_values(['cycle_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Targets built.  Shape: {df.shape}')
target_cols = [
    'cycle_id', 'timestamp', 'stage', 'current_stage',
    'next_stage', 'hours_to_next_stage',
    'transition_within_24h', 'transition_within_48h',
    'stage_progress_fraction',
]
print('\nSample target values:')
display(df[target_cols].head(20))

for cid in df['cycle_id'].unique()[:3]:
    sub = df[df['cycle_id'] == cid]
    print(f'\n── Cycle {cid} ──')
    display(sub[target_cols].head(10))

logger.info(f'Progression targets built | shape={df.shape}')


Building progression targets (may take a moment)...
Targets built.  Shape: (10848, 44)

Sample target values:


,cycle_id,timestamp,stage,current_stage,next_stage,hours_to_next_stage,transition_within_24h,transition_within_48h,stage_progress_fraction
0,1,2024-07-01 00:00:00,seedling,0,1.0,288.0,0.0,0.0,0.000000
1,1,2024-07-01 01:00:00,seedling,0,1.0,287.0,0.0,0.0,0.003484
2,1,2024-07-01 02:00:00,seedling,0,1.0,286.0,0.0,0.0,0.006969
3,1,2024-07-01 03:00:00,seedling,0,1.0,285.0,0.0,0.0,0.010453
4,1,2024-07-01 04:00:00,seedling,0,1.0,284.0,0.0,0.0,0.013937
5,1,2024-07-01 05:00:00,seedling,0,1.0,283.0,0.0,0.0,0.017422
6,1,2024-07-01 06:00:00,seedling,0,1.0,282.0,0.0,0.0,0.020906
7,1,2024-07-01 07:00:00,seedling,0,1.0,281.0,0.0,0.0,0.024390
8,1,2024-07-01 08:00:00,seedling,0,1.0,280.0,0.0,0.0,0.027875
9,1,2024-07-01 09:00:00,seedling,0,1.0,279.0,0.0,0.0,0.031359



── Cycle 1 ──


,cycle_id,timestamp,stage,current_stage,next_stage,hours_to_next_stage,transition_within_24h,transition_within_48h,stage_progress_fraction
0,1,2024-07-01 00:00:00,seedling,0,1.0,288.0,0.0,0.0,0.000000
1,1,2024-07-01 01:00:00,seedling,0,1.0,287.0,0.0,0.0,0.003484
2,1,2024-07-01 02:00:00,seedling,0,1.0,286.0,0.0,0.0,0.006969
3,1,2024-07-01 03:00:00,seedling,0,1.0,285.0,0.0,0.0,0.010453
4,1,2024-07-01 04:00:00,seedling,0,1.0,284.0,0.0,0.0,0.013937
5,1,2024-07-01 05:00:00,seedling,0,1.0,283.0,0.0,0.0,0.017422
6,1,2024-07-01 06:00:00,seedling,0,1.0,282.0,0.0,0.0,0.020906
7,1,2024-07-01 07:00:00,seedling,0,1.0,281.0,0.0,0.0,0.024390
8,1,2024-07-01 08:00:00,seedling,0,1.0,280.0,0.0,0.0,0.027875
9,1,2024-07-01 09:00:00,seedling,0,1.0,279.0,0.0,0.0,0.031359



── Cycle 2 ──


,cycle_id,timestamp,stage,current_stage,next_stage,hours_to_next_stage,transition_within_24h,transition_within_48h,stage_progress_fraction
2712,2,2024-11-10 00:00:00,seedling,0,1.0,288.0,0.0,0.0,0.000000
2713,2,2024-11-10 01:00:00,seedling,0,1.0,287.0,0.0,0.0,0.003484
2714,2,2024-11-10 02:00:00,seedling,0,1.0,286.0,0.0,0.0,0.006969
2715,2,2024-11-10 03:00:00,seedling,0,1.0,285.0,0.0,0.0,0.010453
2716,2,2024-11-10 04:00:00,seedling,0,1.0,284.0,0.0,0.0,0.013937
2717,2,2024-11-10 05:00:00,seedling,0,1.0,283.0,0.0,0.0,0.017422
2718,2,2024-11-10 06:00:00,seedling,0,1.0,282.0,0.0,0.0,0.020906
2719,2,2024-11-10 07:00:00,seedling,0,1.0,281.0,0.0,0.0,0.024390
2720,2,2024-11-10 08:00:00,seedling,0,1.0,280.0,0.0,0.0,0.027875
2721,2,2024-11-10 09:00:00,seedling,0,1.0,279.0,0.0,0.0,0.031359



── Cycle 3 ──


,cycle_id,timestamp,stage,current_stage,next_stage,hours_to_next_stage,transition_within_24h,transition_within_48h,stage_progress_fraction
5328,3,2025-03-05 00:00:00,seedling,0,1.0,408.0,0.0,0.0,0.000000
5329,3,2025-03-05 01:00:00,seedling,0,1.0,407.0,0.0,0.0,0.002457
5330,3,2025-03-05 02:00:00,seedling,0,1.0,406.0,0.0,0.0,0.004914
5331,3,2025-03-05 03:00:00,seedling,0,1.0,405.0,0.0,0.0,0.007371
5332,3,2025-03-05 04:00:00,seedling,0,1.0,404.0,0.0,0.0,0.009828
5333,3,2025-03-05 05:00:00,seedling,0,1.0,403.0,0.0,0.0,0.012285
5334,3,2025-03-05 06:00:00,seedling,0,1.0,402.0,0.0,0.0,0.014742
5335,3,2025-03-05 07:00:00,seedling,0,1.0,401.0,0.0,0.0,0.017199
5336,3,2025-03-05 08:00:00,seedling,0,1.0,400.0,0.0,0.0,0.019656
5337,3,2025-03-05 09:00:00,seedling,0,1.0,399.0,0.0,0.0,0.022113


2026-03-10 19:26:30,300 | INFO | Progression targets built | shape=(10848, 44)


## SECTION 6 — Feature Engineering

In [8]:
# ─────────────────────────────────────────────────────────────
# SECTION 6: Feature Engineering
# ─────────────────────────────────────────────────────────────

# ── 6.1  Identify environmental feature columns ───────────────
ENV_COLS_CANONICAL = ['temperature', 'humidity', 'light', 'soil_moisture', 'co2']
EXCLUDE_COLS = {
    'timestamp', 'cycle_id', 'stage', 'stage_int',
    'current_stage', 'next_stage', 'hours_to_next_stage',
    'transition_within_24h', 'transition_within_48h',
    'stage_progress_fraction',
}

env_cols = [c for c in df.columns if c not in EXCLUDE_COLS
            and pd.api.types.is_numeric_dtype(df[c])]
print('Raw environmental features:', env_cols)


# ── 6.2  Feature engineering function (per cycle) ─────────────
ROLLING_WINDOWS = [6, 12, 24]
LAG_STEPS       = [1, 2, 3, 6, 12]


def engineer_features_for_cycle(
    grp: pd.DataFrame,
    raw_cols: list,
    rolling_windows: list,
    lag_steps: list,
) -> pd.DataFrame:
    """Derive temporal features within a single cycle (no data leakage)."""
    grp = grp.sort_values('timestamp').copy()

    t0 = grp['timestamp'].min()
    grp['elapsed_hours'] = (grp['timestamp'] - t0).dt.total_seconds() / 3600.0
    grp['hour_of_day']   = grp['timestamp'].dt.hour
    grp['day_of_cycle']  = (grp['elapsed_hours'] / 24).astype(int)

    for col in raw_cols:
        if col not in grp.columns:
            continue
        for w in rolling_windows:
            grp[f'{col}_roll_mean_{w}h'] = (
                grp[col].rolling(window=w, min_periods=1).mean()
            )
            grp[f'{col}_roll_std_{w}h'] = (
                grp[col].rolling(window=w, min_periods=1).std().fillna(0)
            )
        for lag in lag_steps:
            grp[f'{col}_lag_{lag}'] = grp[col].shift(lag)

    if 'light' in grp.columns:
        grp['cumulative_light'] = grp['light'].expanding().sum()

    if 'temperature' in grp.columns:
        GDH_BASE = 10.0
        grp['gdh']            = (grp['temperature'] - GDH_BASE).clip(lower=0)
        grp['cumulative_gdh'] = grp['gdh'].expanding().sum()

    if 'temperature' in grp.columns and 'humidity' in grp.columns:
        T  = grp['temperature']
        RH = grp['humidity']
        es = 0.6108 * np.exp(17.27 * T / (T + 237.3))
        grp['vpd'] = es * (1 - RH / 100.0)

    return grp


print('Engineering features per cycle...')
engineered_chunks = [
    engineer_features_for_cycle(grp, env_cols, ROLLING_WINDOWS, LAG_STEPS)
    for _, grp in df.groupby('cycle_id', sort=False)
]
df_feat = pd.concat(engineered_chunks, ignore_index=True)
df_feat.sort_values(['cycle_id', 'timestamp'], inplace=True)
df_feat.reset_index(drop=True, inplace=True)

# ── 6.3  Identify all feature columns ─────────────────────────
NON_FEATURE_COLS = list(EXCLUDE_COLS) + ['stage_progress_fraction']
FEATURE_COLS = [
    c for c in df_feat.columns
    if c not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(df_feat[c])
]
print(f'\nTotal features: {len(FEATURE_COLS)}')
print('Features:', FEATURE_COLS)

df_feat[FEATURE_COLS] = df_feat[FEATURE_COLS].bfill().fillna(0)

print(f'\nFinal engineered DataFrame shape: {df_feat.shape}')
display(df_feat[FEATURE_COLS[:8]].head(4))
logger.info(f'Feature engineering done | n_features={len(FEATURE_COLS)} df_shape={df_feat.shape}')

# ── Save feature list to artifacts ────────────────────────────
feature_info = {
    'n_features'    : len(FEATURE_COLS),
    'feature_cols'  : FEATURE_COLS,
    'rolling_windows': ROLLING_WINDOWS,
    'lag_steps'     : LAG_STEPS,
    'env_cols'      : env_cols,
}
with open(os.path.join(METRICS_DIR, 'feature_list.json'), 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f'Feature list saved.')


Raw environmental features: ['year', 'month', 'day_of_year', 'week_of_year', 'hour', 'days_from_cycle_start', 'stage_index', 'hours_in_current_stage', 'days_in_current_stage', 'stage_duration_hours', 'stage_duration_days', 'stage_progress_pct', 'total_cycle_progress_pct', 'estimated_days_to_next_stage', 'estimated_hours_to_next_stage', 'is_stage_transition', 'indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_co2', 'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy', 'light_period_flag', 'cumulative_gdd_like_index']
Engineering features per cycle...

Total features: 358
Features: ['year', 'month', 'day_of_year', 'week_of_year', 'hour', 'days_from_cycle_start', 'stage_index', 'hours_in_current_stage', 'days_in_current_stage', 'stage_duration_hours', 'stage_duration_days', 'stage_progress_pct', 'total_cycle_progress_pct', 'estimated_days_to_next_stage', 'estimated_hours_to_next

,year,month,day_of_year,week_of_year,hour,days_from_cycle_start,stage_index,hours_in_current_stage
0,2024,7,183,27,0,0.0000,0,0.0
1,2024,7,183,27,1,0.0417,0,1.0
2,2024,7,183,27,2,0.0833,0,2.0
3,2024,7,183,27,3,0.1250,0,3.0


2026-03-10 19:26:40,369 | INFO | Feature engineering done | n_features=358 df_shape=(10848, 377)


Feature list saved.


## SECTION 7 — Sequence Building for LSTM

In [9]:
# ─────────────────────────────────────────────────────────────
# SECTION 7: Sequence Building
# ─────────────────────────────────────────────────────────────

SEQ_LEN = 24   # number of timesteps per input window


def build_sequences_for_cycle(
    cycle_df: pd.DataFrame,
    feature_cols: list,
    seq_len: int,
) -> tuple:
    """Slide a window of `seq_len` over a single cycle. Returns all targets."""
    cdf = cycle_df.sort_values('timestamp').reset_index(drop=True)
    n   = len(cdf)
    if n < seq_len:
        return None

    X_list, y_cur, y_nxt, y_hrs, y_t24, y_t48 = [], [], [], [], [], []

    feat_vals = cdf[feature_cols].values.astype(np.float32)
    cur_stage = cdf['current_stage'].values
    nxt_stage = cdf['next_stage'].values
    hrs       = cdf['hours_to_next_stage'].values
    t24       = cdf['transition_within_24h'].values
    t48       = cdf['transition_within_48h'].values

    for end in range(seq_len - 1, n):
        start = end - seq_len + 1
        X_list.append(feat_vals[start: end + 1])
        y_cur.append(cur_stage[end])
        y_nxt.append(nxt_stage[end])
        y_hrs.append(hrs[end])
        y_t24.append(t24[end])
        y_t48.append(t48[end])

    return (
        np.stack(X_list),
        np.array(y_cur, dtype=np.int32),
        np.array(y_nxt, dtype=np.float32),
        np.array(y_hrs, dtype=np.float32),
        np.array(y_t24, dtype=np.float32),
        np.array(y_t48, dtype=np.float32),
    )


def build_all_sequences(df_in, feature_cols, seq_len):
    Xs, ycs, ynxts, yhrs, yt24s, yt48s = [], [], [], [], [], []
    cycle_ids_out = []

    for cid, grp in df_in.groupby('cycle_id', sort=False):
        result = build_sequences_for_cycle(grp, feature_cols, seq_len)
        if result is None:
            continue
        X, yc, yn, yh, yt24, yt48 = result
        Xs.append(X);       ycs.append(yc);    ynxts.append(yn)
        yhrs.append(yh);    yt24s.append(yt24); yt48s.append(yt48)
        cycle_ids_out.extend([cid] * len(yc))

    return (
        np.concatenate(Xs),
        np.concatenate(ycs),
        np.concatenate(ynxts),
        np.concatenate(yhrs),
        np.concatenate(yt24s),
        np.concatenate(yt48s),
        np.array(cycle_ids_out),
    )


print(f'Building sequences (window={SEQ_LEN}) ...')
(
    X_all, y_cur_all, y_nxt_all,
    y_hrs_all, y_t24_all, y_t48_all,
    cycle_ids_all
) = build_all_sequences(df_feat, FEATURE_COLS, SEQ_LEN)

print(f'\nTensor shapes:')
print(f'  X               : {X_all.shape}   [samples, timesteps, features]')
print(f'  y_current_stage : {y_cur_all.shape}')
print(f'  y_next_stage    : {y_nxt_all.shape}')
print(f'  y_hours_to_next : {y_hrs_all.shape}')
print(f'  y_trans_24h     : {y_t24_all.shape}')
print(f'  y_trans_48h     : {y_t48_all.shape}')
logger.info(f'Sequences built | total_samples={X_all.shape[0]} seq_len={SEQ_LEN}')


Building sequences (window=24) ...


2026-03-10 19:26:43,853 | INFO | Sequences built | total_samples=10756 seq_len=24



Tensor shapes:
  X               : (10756, 24, 358)   [samples, timesteps, features]
  y_current_stage : (10756,)
  y_next_stage    : (10756,)
  y_hours_to_next : (10756,)
  y_trans_24h     : (10756,)
  y_trans_48h     : (10756,)


## SECTION 8 — Train / Validation / Test Split by Cycle

In [10]:
# ─────────────────────────────────────────────────────────────
# SECTION 8: Train / Val / Test Split  (cycle-level, no leakage)
# ─────────────────────────────────────────────────────────────

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# TEST_FRAC  = 0.15  (remainder)

unique_cycles = np.array(sorted(df_feat['cycle_id'].unique()))
np.random.shuffle(unique_cycles)

n_cycles = len(unique_cycles)
n_train  = max(1, int(n_cycles * TRAIN_FRAC))
n_val    = max(1, int(n_cycles * VAL_FRAC))
n_test   = n_cycles - n_train - n_val
if n_test < 1:
    n_val  -= 1
    n_test  = 1

train_cycles = set(unique_cycles[:n_train])
val_cycles   = set(unique_cycles[n_train: n_train + n_val])
test_cycles  = set(unique_cycles[n_train + n_val:])

print(f'Total cycles : {n_cycles}')
print(f'Train cycles : {len(train_cycles)}  → {sorted(train_cycles)}')
print(f'Val   cycles : {len(val_cycles)}    → {sorted(val_cycles)}')
print(f'Test  cycles : {len(test_cycles)}   → {sorted(test_cycles)}')

train_mask = np.isin(cycle_ids_all, list(train_cycles))
val_mask   = np.isin(cycle_ids_all, list(val_cycles))
test_mask  = np.isin(cycle_ids_all, list(test_cycles))


def apply_mask(mask):
    return (
        X_all[mask],
        y_cur_all[mask],
        y_nxt_all[mask],
        y_hrs_all[mask],
        y_t24_all[mask],
        y_t48_all[mask],
    )


X_tr, y_cur_tr, y_nxt_tr, y_hrs_tr, y_t24_tr, y_t48_tr = apply_mask(train_mask)
X_va, y_cur_va, y_nxt_va, y_hrs_va, y_t24_va, y_t48_va = apply_mask(val_mask)
X_te, y_cur_te, y_nxt_te, y_hrs_te, y_t24_te, y_t48_te = apply_mask(test_mask)

print(f'\nSamples per split:')
print(f'  Train : {X_tr.shape[0]}')
print(f'  Val   : {X_va.shape[0]}')
print(f'  Test  : {X_te.shape[0]}')
logger.info(
    f'Split | train_cycles={len(train_cycles)} val_cycles={len(val_cycles)} '
    f'test_cycles={len(test_cycles)} '
    f'train_samples={X_tr.shape[0]} val_samples={X_va.shape[0]} test_samples={X_te.shape[0]}'
)

# ── Save split summary ────────────────────────────────────────
split_summary = {
    'train_frac'    : TRAIN_FRAC,
    'val_frac'      : VAL_FRAC,
    'n_total_cycles': n_cycles,
    'train_cycles'  : sorted([str(c) for c in train_cycles]),
    'val_cycles'    : sorted([str(c) for c in val_cycles]),
    'test_cycles'   : sorted([str(c) for c in test_cycles]),
    'train_samples' : int(X_tr.shape[0]),
    'val_samples'   : int(X_va.shape[0]),
    'test_samples'  : int(X_te.shape[0]),
    'seq_len'       : SEQ_LEN,
    'n_features'    : int(X_tr.shape[2]),
}
with open(os.path.join(METRICS_DIR, 'split_summary.json'), 'w') as f:
    json.dump(split_summary, f, indent=2)
print(f'\nSplit summary saved.')


2026-03-10 19:26:47,774 | INFO | Split | train_cycles=2 val_cycles=1 test_cycles=1 train_samples=5282 val_samples=2689 test_samples=2785


Total cycles : 4
Train cycles : 2  → [np.int64(2), np.int64(4)]
Val   cycles : 1    → [np.int64(1)]
Test  cycles : 1   → [np.int64(3)]

Samples per split:
  Train : 5282
  Val   : 2689
  Test  : 2785

Split summary saved.


## SECTION 9 — Scaling

In [11]:
# ─────────────────────────────────────────────────────────────
# SECTION 9: Scaling
# ─────────────────────────────────────────────────────────────

n_features = X_tr.shape[2]

scaler = StandardScaler()
X_tr_2d = X_tr.reshape(-1, n_features)
scaler.fit(X_tr_2d)   # fit ONLY on training data


def scale_3d(X, scaler):
    s, t, f = X.shape
    return scaler.transform(X.reshape(-1, f)).reshape(s, t, f).astype(np.float32)


X_tr_sc = scale_3d(X_tr, scaler)
X_va_sc = scale_3d(X_va, scaler)
X_te_sc = scale_3d(X_te, scaler)

print(f'X_tr scaled shape: {X_tr_sc.shape}')

# ── Scale regression target (hours_to_next_stage) ─────────────
hrs_train_valid = y_hrs_tr[~np.isnan(y_hrs_tr)]
HRS_MEAN = hrs_train_valid.mean()
HRS_STD  = hrs_train_valid.std() + 1e-8


def scale_hours(arr):
    out = arr.copy()
    out[~np.isnan(out)] = (out[~np.isnan(out)] - HRS_MEAN) / HRS_STD
    return out


y_hrs_tr_sc = scale_hours(y_hrs_tr)
y_hrs_va_sc = scale_hours(y_hrs_va)
y_hrs_te_sc = scale_hours(y_hrs_te)

print(f'Hours-to-next scaler  mean={HRS_MEAN:.2f}h  std={HRS_STD:.2f}h')
print('\n✅ Scaling complete.')
logger.info(f'Scaling done | hrs_mean={HRS_MEAN:.2f} hrs_std={HRS_STD:.2f}')

# ── Save scaler details ────────────────────────────────────────
scaler_details = {
    'scaler_type'  : 'StandardScaler',
    'n_features'   : n_features,
    'feature_cols' : FEATURE_COLS,
    'mean'         : scaler.mean_.tolist(),
    'scale'        : scaler.scale_.tolist(),
    'var'          : scaler.var_.tolist(),
    'hrs_mean'     : float(HRS_MEAN),
    'hrs_std'      : float(HRS_STD),
}
with open(os.path.join(METRICS_DIR, 'scaler_details.json'), 'w') as f:
    json.dump(scaler_details, f, indent=2)
print('Scaler details saved.')


2026-03-10 19:26:53,372 | INFO | Scaling done | hrs_mean=270.33 hrs_std=192.35


X_tr scaled shape: (5282, 24, 358)
Hours-to-next scaler  mean=270.33h  std=192.35h

✅ Scaling complete.
Scaler details saved.


## SECTION 10 — Baseline Models (Random Forest / XGBoost)

In [12]:
# ─────────────────────────────────────────────────────────────
# SECTION 10: Baseline Models
# ─────────────────────────────────────────────────────────────

# Use last timestep of each sequence as a flat feature vector
X_tr_flat = X_tr_sc[:, -1, :]
X_va_flat = X_va_sc[:, -1, :]
X_te_flat = X_te_sc[:, -1, :]

# ── 10.1  Current stage classifier (RF) ──────────────────────
print('Training RF baseline — current stage classifier...')
rf_cls = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
rf_cls.fit(X_tr_flat, y_cur_tr)
rf_cls_preds = rf_cls.predict(X_te_flat)

rf_acc = accuracy_score(y_cur_te, rf_cls_preds)
print(f'Baseline current-stage accuracy: {rf_acc:.4f}')
rf_report_str = classification_report(y_cur_te, rf_cls_preds, target_names=STAGE_ORDER, zero_division=0)
print(rf_report_str)

# ── 10.2  Hours-to-next regressor (XGBoost) ──────────────────
valid_mask_tr = ~np.isnan(y_hrs_tr_sc)
valid_mask_te = ~np.isnan(y_hrs_te_sc)

print('Training XGBoost baseline — hours-to-next-stage regressor...')
xgb_reg = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    random_state=RANDOM_SEED, n_jobs=-1, verbosity=0,
)
xgb_reg.fit(
    X_tr_flat[valid_mask_tr],
    y_hrs_tr_sc[valid_mask_tr],
    eval_set=[(
        X_va_flat[~np.isnan(y_hrs_va_sc)],
        y_hrs_va_sc[~np.isnan(y_hrs_va_sc)],
    )],
    verbose=False,
)

xgb_preds_sc  = xgb_reg.predict(X_te_flat[valid_mask_te])
xgb_preds_hrs = xgb_preds_sc * HRS_STD + HRS_MEAN
actual_hrs_te = y_hrs_te[valid_mask_te]

mae_bl  = mean_absolute_error(actual_hrs_te, xgb_preds_hrs)
rmse_bl = np.sqrt(mean_squared_error(actual_hrs_te, xgb_preds_hrs))
print(f'Baseline hours-to-next  MAE={mae_bl:.2f}h  RMSE={rmse_bl:.2f}h')

# ── 10.3  Feature importances ─────────────────────────────────
feat_imp = pd.Series(rf_cls.feature_importances_, index=FEATURE_COLS)
print('\nTop-15 RF feature importances (current stage):')
print(feat_imp.nlargest(15))

# ── Save baseline artifacts ────────────────────────────────────
baseline_metrics = {
    'rf_current_stage_accuracy': float(rf_acc),
    'rf_classification_report' : rf_report_str,
    'xgb_hours_mae'            : float(mae_bl),
    'xgb_hours_rmse'           : float(rmse_bl),
}
with open(os.path.join(METRICS_DIR, 'baseline_metrics.json'), 'w') as f:
    json.dump(baseline_metrics, f, indent=2)

feat_imp.nlargest(30).reset_index().rename(
    columns={'index': 'feature', 0: 'importance'}
).to_csv(os.path.join(METRICS_DIR, 'rf_feature_importances.csv'), index=False)

with open(os.path.join(REPORTS_DIR, 'baseline_rf_classification_report.txt'), 'w', encoding='utf-8') as f:
    f.write(rf_report_str)

logger.info(f'Baselines | RF_acc={rf_acc:.4f} XGB_MAE={mae_bl:.2f}h RMSE={rmse_bl:.2f}h')
print('\n✅ Baseline artifacts saved.')


Training RF baseline — current stage classifier...
Baseline current-stage accuracy: 1.0000
                      precision    recall  f1-score   support

            seedling       1.00      1.00      1.00       385
    early_vegetative       1.00      1.00      1.00       624
flowering_initiation       1.00      1.00      1.00       288
           flowering       1.00      1.00      1.00       456
              unripe       1.00      1.00      1.00       792
                ripe       1.00      1.00      1.00       240

            accuracy                           1.00      2785
           macro avg       1.00      1.00      1.00      2785
        weighted avg       1.00      1.00      1.00      2785

Training XGBoost baseline — hours-to-next-stage regressor...


2026-03-10 19:27:10,186 | INFO | Baselines | RF_acc=1.0000 XGB_MAE=1.56h RMSE=7.73h


Baseline hours-to-next  MAE=1.56h  RMSE=7.73h

Top-15 RF feature importances (current stage):
stage_duration_days                   0.041610
stage_duration_hours_lag_1            0.037940
stage_duration_days_roll_mean_6h      0.037699
stage_duration_days_roll_mean_12h     0.037038
stage_duration_hours_lag_6            0.036882
stage_duration_hours_lag_3            0.033873
stage_duration_days_lag_1             0.032093
stage_progress_pct_roll_std_6h        0.031538
stage_duration_days_lag_2             0.029772
stage_duration_hours                  0.029069
stage_duration_hours_roll_mean_12h    0.027382
stage_index_roll_mean_12h             0.027354
stage_index_lag_2                     0.026308
stage_index_roll_mean_24h             0.025980
total_cycle_progress_pct_lag_3        0.025045
dtype: float64

✅ Baseline artifacts saved.


## SECTION 11 — Main LSTM Multi-Task Model

In [13]:
# ─────────────────────────────────────────────────────────────
# SECTION 11: LSTM Multi-Task Model
# ─────────────────────────────────────────────────────────────

# ── 11.1  Mask helper for NaN targets ────────────────────────
def nan_mask_weights(arr: np.ndarray) -> np.ndarray:
    return (~np.isnan(arr)).astype(np.float32)


def fill_nan(arr, fill=0.0):
    out = arr.copy()
    out[np.isnan(out)] = fill
    return out


sw_nxt_tr = nan_mask_weights(y_nxt_tr)
sw_hrs_tr = nan_mask_weights(y_hrs_tr_sc)
sw_t24_tr = nan_mask_weights(y_t24_tr)
sw_t48_tr = nan_mask_weights(y_t48_tr)

sw_nxt_va = nan_mask_weights(y_nxt_va)
sw_hrs_va = nan_mask_weights(y_hrs_va_sc)
sw_t24_va = nan_mask_weights(y_t24_va)
sw_t48_va = nan_mask_weights(y_t48_va)

y_nxt_tr_f = fill_nan(y_nxt_tr).astype(np.int32)
y_hrs_tr_f = fill_nan(y_hrs_tr_sc).astype(np.float32)
y_t24_tr_f = fill_nan(y_t24_tr).astype(np.float32)
y_t48_tr_f = fill_nan(y_t48_tr).astype(np.float32)

y_nxt_va_f = fill_nan(y_nxt_va).astype(np.int32)
y_hrs_va_f = fill_nan(y_hrs_va_sc).astype(np.float32)
y_t24_va_f = fill_nan(y_t24_va).astype(np.float32)
y_t48_va_f = fill_nan(y_t48_va).astype(np.float32)


# ── 11.2  Class weights for imbalanced stages ─────────────────
def get_class_weights(y: np.ndarray, n_classes: int) -> dict:
    classes = np.arange(n_classes)
    weights = compute_class_weight(
        class_weight='balanced', classes=classes, y=y
    )
    return {i: float(w) for i, w in enumerate(weights)}


cw_cur = get_class_weights(y_cur_tr, N_STAGES)
print('Current-stage class weights:', cw_cur)

# ── Save class weights ─────────────────────────────────────────
stage_dist_tr = {
    STAGE_ORDER[i]: int((y_cur_tr == i).sum()) for i in range(N_STAGES)
}
class_info = {
    'class_weights'       : {str(k): v for k, v in cw_cur.items()},
    'stage_distribution_train': stage_dist_tr,
    'stage_names'         : STAGE_ORDER,
}
with open(os.path.join(METRICS_DIR, 'class_weights.json'), 'w') as f:
    json.dump(class_info, f, indent=2)
print('Class weights saved.')


# ── 11.3  Build LSTM model ────────────────────────────────────
LSTM_UNITS_1  = 128
LSTM_UNITS_2  = 64
DROPOUT_RATE  = 0.3
DENSE_UNITS   = 64


def build_lstm_model(
    seq_len: int,
    n_features: int,
    n_stages: int,
    lstm_units_1: int = LSTM_UNITS_1,
    lstm_units_2: int = LSTM_UNITS_2,
    dropout_rate: float = DROPOUT_RATE,
    dense_units: int = DENSE_UNITS,
) -> keras.Model:
    """
    Shared LSTM backbone → 5 output heads:
      1. current_stage   (softmax, N_STAGES)
      2. next_stage      (softmax, N_STAGES)
      3. hours_to_next   (linear, 1)
      4. trans_24h       (sigmoid, 1)
      5. trans_48h       (sigmoid, 1)
    """
    inp = Input(shape=(seq_len, n_features), name='sequence_input')

    x = layers.LSTM(
        lstm_units_1, return_sequences=True,
        dropout=dropout_rate, recurrent_dropout=0.0, name='lstm_1',
    )(inp)
    x = layers.LSTM(
        lstm_units_2, return_sequences=False,
        dropout=dropout_rate, recurrent_dropout=0.0, name='lstm_2',
    )(x)
    x = layers.BatchNormalization(name='bn_shared')(x)
    x = layers.Dense(dense_units, activation='relu', name='shared_dense')(x)
    x = layers.Dropout(dropout_rate, name='dropout_shared')(x)

    h1     = layers.Dense(32, activation='relu', name='h1_dense')(x)
    out_cur = layers.Dense(n_stages, activation='softmax', name='current_stage')(h1)

    h2     = layers.Dense(32, activation='relu', name='h2_dense')(x)
    out_nxt = layers.Dense(n_stages, activation='softmax', name='next_stage')(h2)

    h3     = layers.Dense(32, activation='relu', name='h3_dense')(x)
    out_hrs = layers.Dense(1, activation='linear', name='hours_to_next')(h3)

    h4     = layers.Dense(16, activation='relu', name='h4_dense')(x)
    out_t24 = layers.Dense(1, activation='sigmoid', name='trans_24h')(h4)

    h5     = layers.Dense(16, activation='relu', name='h5_dense')(x)
    out_t48 = layers.Dense(1, activation='sigmoid', name='trans_48h')(h5)

    model = Model(
        inputs=inp,
        outputs=[out_cur, out_nxt, out_hrs, out_t24, out_t48],
        name='TomatoProgressionModel',
    )
    return model


model = build_lstm_model(seq_len=SEQ_LEN, n_features=n_features, n_stages=N_STAGES)

# ── 11.4  Compile ─────────────────────────────────────────────
LOSS_WEIGHTS = {
    'current_stage': 2.0,
    'next_stage'   : 2.0,
    'hours_to_next': 1.0,
    'trans_24h'    : 1.0,
    'trans_48h'    : 1.0,
}

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'current_stage': 'sparse_categorical_crossentropy',
        'next_stage'   : 'sparse_categorical_crossentropy',
        'hours_to_next': keras.losses.Huber(delta=1.0),
        'trans_24h'    : 'binary_crossentropy',
        'trans_48h'    : 'binary_crossentropy',
    },
    loss_weights=LOSS_WEIGHTS,
    metrics={
        'current_stage': 'accuracy',
        'next_stage'   : 'accuracy',
        'hours_to_next': keras.metrics.MeanAbsoluteError(name='mae'),
        'trans_24h'    : 'accuracy',
        'trans_48h'    : 'accuracy',
    },
)

model.summary()

# ── Save model config / hyperparameters ───────────────────────
model_config = {
    'run_id'        : RUN_ID,
    'architecture'  : 'LSTM_MultiTask',
    'seq_len'       : SEQ_LEN,
    'n_features'    : n_features,
    'n_stages'      : N_STAGES,
    'lstm_units_1'  : LSTM_UNITS_1,
    'lstm_units_2'  : LSTM_UNITS_2,
    'dropout_rate'  : DROPOUT_RATE,
    'dense_units'   : DENSE_UNITS,
    'loss_weights'  : LOSS_WEIGHTS,
    'optimizer'     : 'Adam',
    'learning_rate' : 1e-3,
    'stage_order'   : STAGE_ORDER,
    'stage_to_int'  : STAGE_TO_INT,
    'int_to_stage'  : {str(k): v for k, v in INT_TO_STAGE.items()},
    'feature_cols'  : FEATURE_COLS,
    'hrs_mean'      : float(HRS_MEAN),
    'hrs_std'       : float(HRS_STD),
}
with open(os.path.join(METRICS_DIR, 'model_config.json'), 'w') as f:
    json.dump(model_config, f, indent=2)
print('\nModel config saved.')
logger.info('Model built and config saved.')


Current-stage class weights: {0: 1.3543589743589743, 1: 0.7336111111111111, 2: 1.8340277777777778, 3: 1.2226851851851852, 4: 0.5822310405643739, 5: 1.2226851851851852}
Class weights saved.


Model: "TomatoProgressionModel"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 24, 358)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 24, 128)   │    249,344 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     49,408 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_shared           │ (None, 64)        │        256 │ lstm_2[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense        │ (None, 64)        │      4,160 │ bn_shared[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_shared      │ (None, 64)        │          0 │ shared_dense[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h1_dense (Dense)    │ (None, 32)        │      2,080 │ dropout_shared[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h2_dense (Dense)    │ (None, 32)        │      2,080 │ dropout_shared[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h3_dense (Dense)    │ (None, 32)        │      2,080 │ dropout_shared[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h4_dense (Dense)    │ (None, 16)        │      1,040 │ dropout_shared[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ h5_dense (Dense)    │ (None, 16)        │      1,040 │ dropout_shared[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ current_stage       │ (None, 6)         │        198 │ h1_dense[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ next_stage (Dense)  │ (None, 6)         │        198 │ h2_dense[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hours_to_next       │ (None, 1)         │         33 │ h3_dense[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ trans_24h (Dense)   │ (None, 1)         │         17 │ h4_dense[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ trans_48h (Dense)   │ (None, 1)         │         17 │ h5_dense[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 311,951 (1.19 MB)

 Trainable params: 311,823 (1.19 MB)

 Non-trainable params: 128 (512.00 B)

2026-03-10 19:27:19,271 | INFO | Model built and config saved.



Model config saved.


## SECTION 12 — Training

In [14]:
# ─────────────────────────────────────────────────────────────
# SECTION 12: Training
# ─────────────────────────────────────────────────────────────

EPOCHS     = 150
BATCH_SIZE = 64

cb_early = EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True, verbose=1,
)
cb_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=7,
    min_lr=1e-6, verbose=1,
)
cb_ckpt = ModelCheckpoint(
    CKPT_PATH, monitor='val_loss',
    save_best_only=True, verbose=0,
)

# ── Ensure correct dtypes/shapes ─────────────────────────────
X_tr_sc = np.asarray(X_tr_sc, dtype=np.float32)
X_va_sc = np.asarray(X_va_sc, dtype=np.float32)

y_cur_tr   = np.asarray(y_cur_tr,   dtype=np.int32).reshape(-1)
y_nxt_tr_f = np.asarray(y_nxt_tr_f, dtype=np.int32).reshape(-1)
y_hrs_tr_f = np.asarray(y_hrs_tr_f, dtype=np.float32).reshape(-1, 1)
y_t24_tr_f = np.asarray(y_t24_tr_f, dtype=np.float32).reshape(-1, 1)
y_t48_tr_f = np.asarray(y_t48_tr_f, dtype=np.float32).reshape(-1, 1)

y_cur_va   = np.asarray(y_cur_va,   dtype=np.int32).reshape(-1)
y_nxt_va_f = np.asarray(y_nxt_va_f, dtype=np.int32).reshape(-1)
y_hrs_va_f = np.asarray(y_hrs_va_f, dtype=np.float32).reshape(-1, 1)
y_t24_va_f = np.asarray(y_t24_va_f, dtype=np.float32).reshape(-1, 1)
y_t48_va_f = np.asarray(y_t48_va_f, dtype=np.float32).reshape(-1, 1)

print('Model output names:', model.output_names)
print('X_tr_sc:', X_tr_sc.shape, X_tr_sc.dtype)
print('X_va_sc:', X_va_sc.shape, X_va_sc.dtype)

# ── Build tf.data datasets ────────────────────────────────────
train_ds = tf.data.Dataset.from_tensor_slices((
    X_tr_sc,
    {
        'current_stage': y_cur_tr,
        'next_stage'   : y_nxt_tr_f,
        'hours_to_next': y_hrs_tr_f,
        'trans_24h'    : y_t24_tr_f,
        'trans_48h'    : y_t48_tr_f,
    }
))
val_ds = tf.data.Dataset.from_tensor_slices((
    X_va_sc,
    {
        'current_stage': y_cur_va,
        'next_stage'   : y_nxt_va_f,
        'hours_to_next': y_hrs_va_f,
        'trans_24h'    : y_t24_va_f,
        'trans_48h'    : y_t48_va_f,
    }
))
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f'Training on {X_tr_sc.shape[0]} samples  |  Validating on {X_va_sc.shape[0]} samples')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
logger.info(f'Training started | epochs={EPOCHS} batch={BATCH_SIZE}')

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[cb_early, cb_lr, cb_ckpt],
    verbose=1,
)
logger.info(f'Training complete | epochs_run={len(history.history["loss"])}')

# ── Save training history ─────────────────────────────────────
hist_df = pd.DataFrame(history.history)
hist_df.index.name = 'epoch'
hist_df.to_csv(os.path.join(METRICS_DIR, 'training_history.csv'))
with open(os.path.join(METRICS_DIR, 'training_history.json'), 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)
print('Training history saved.')

# ── Plot training curves ──────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
plot_pairs = [
    ('loss',                     'val_loss',                     'Total Loss'),
    ('current_stage_accuracy',   'val_current_stage_accuracy',   'Current Stage Accuracy'),
    ('next_stage_accuracy',      'val_next_stage_accuracy',      'Next Stage Accuracy'),
    ('hours_to_next_mae',        'val_hours_to_next_mae',        'Hours MAE'),
    ('trans_24h_accuracy',       'val_trans_24h_accuracy',       '24h Transition Accuracy'),
    ('trans_48h_accuracy',       'val_trans_48h_accuracy',       '48h Transition Accuracy'),
]
for ax, (tr_key, va_key, title) in zip(axes.flat, plot_pairs):
    if tr_key in history.history:
        ax.plot(history.history[tr_key], label='Train')
        if va_key in history.history:
            ax.plot(history.history[va_key], label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()
plt.suptitle('Training History', fontsize=14)
plt.tight_layout()
p = save_fig(fig, 'training_history_curves')
print(f'Training curves saved → {p}')
plt.show()
plt.close(fig)


Model output names: ListWrapper(['current_stage', 'next_stage', 'hours_to_next', 'trans_24h', 'trans_48h'])
X_tr_sc: (5282, 24, 358) float32
X_va_sc: (2689, 24, 358) float32


2026-03-10 19:27:34,643 | INFO | Training started | epochs=150 batch=64


Training on 5282 samples  |  Validating on 2689 samples
GPU: []
Epoch 1/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 31s 145ms/step - current_stage_accuracy: 0.1414 - current_stage_loss: 2.0267 - hours_to_next_loss: 0.4608 - hours_to_next_mae: 0.8375 - loss: 9.3315 - next_stage_accuracy: 0.1280 - next_stage_loss: 2.0463 - trans_24h_accuracy: 0.9135 - trans_24h_loss: 0.3134 - trans_48h_accuracy: 0.8470 - trans_48h_loss: 0.4091 - val_current_stage_accuracy: 0.2741 - val_current_stage_loss: 1.7256 - val_hours_to_next_loss: 0.3196 - val_hours_to_next_mae: 0.6662 - val_loss: 7.8935 - val_next_stage_accuracy: 0.3909 - val_next_stage_loss: 1.7125 - val_trans_24h_accuracy: 0.9554 - val_trans_24h_loss: 0.3231 - val_trans_48h_accuracy: 0.9107 - val_trans_48h_loss: 0.3706 - learning_rate: 0.0010
Epoch 2/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 10s 119ms/step - current_stage_accuracy: 0.2172 - current_stage_loss: 1.8005 - hours_to_next_loss: 0.3926 - hours_to_next_mae: 0.7541 - loss: 8.0580 - next_stage_accuracy: 0.2331 

2026-03-10 19:33:49,042 | INFO | Training complete | epochs_run=35


Training history saved.
Training curves saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\training_history_curves.png


## SECTION 13 — Evaluation on Test Set

In [16]:
# ─────────────────────────────────────────────────────────────
# SECTION 13: Evaluation
# ─────────────────────────────────────────────────────────────

preds = model.predict(X_te_sc, batch_size=128, verbose=0)
p_cur, p_nxt, p_hrs_sc, p_t24, p_t48 = preds

pred_cur_labels = np.argmax(p_cur, axis=1)
pred_nxt_labels = np.argmax(p_nxt, axis=1)
pred_hrs_raw    = p_hrs_sc[:, 0] * HRS_STD + HRS_MEAN
pred_t24_prob   = p_t24[:, 0]
pred_t48_prob   = p_t48[:, 0]
pred_t24_binary = (pred_t24_prob >= 0.5).astype(int)
pred_t48_binary = (pred_t48_prob >= 0.5).astype(int)

lstm_metrics = {}   # collector

# ── 13.1  Stage classification reports ───────────────────────
for name, true, pred, key in [
    ('CURRENT STAGE', y_cur_te, pred_cur_labels, 'current_stage'),
    ('NEXT STAGE (valid rows only)',
     y_nxt_te[~np.isnan(y_nxt_te)].astype(int),
     pred_nxt_labels[~np.isnan(y_nxt_te)],
     'next_stage'),
]:
    print(f'\n── {name} ──')
    labels_present = sorted(np.unique(np.concatenate([true, pred])))
    tgt_names = [INT_TO_STAGE[l] for l in labels_present]
    report_str = classification_report(true, pred, labels=labels_present,
                                       target_names=tgt_names, zero_division=0)
    print(report_str)
    lstm_metrics[f'{key}_accuracy'] = float(accuracy_score(true, pred))

    with open(os.path.join(REPORTS_DIR, f'lstm_{key}_classification_report.txt'), 'w', encoding='utf-8') as f:
        f.write(f'── {name} ──\n{report_str}')

    cm = confusion_matrix(true, pred, labels=labels_present)
    fig, ax = plt.subplots(figsize=(7, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=tgt_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {name}')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    p = save_fig(fig, f'confusion_matrix_{key}')
    print(f'Confusion matrix saved → {p}')
    plt.show()
    plt.close(fig)

# ── 13.2  Hours regression metrics ───────────────────────────
valid_hrs = ~np.isnan(y_hrs_te)
ah = y_hrs_te[valid_hrs]
ph = pred_hrs_raw[valid_hrs]

mae_lstm  = mean_absolute_error(ah, ph)
rmse_lstm = np.sqrt(mean_squared_error(ah, ph))
nonzero   = ah != 0
mape_lstm = np.mean(np.abs((ah[nonzero] - ph[nonzero]) / ah[nonzero])) * 100

lstm_metrics.update({'hours_mae': float(mae_lstm), 'hours_rmse': float(rmse_lstm),
                     'hours_mape_pct': float(mape_lstm)})

print(f'\n── Hours-to-next regression ──')
print(f'  MAE  : {mae_lstm:.2f} h   (Baseline: {mae_bl:.2f} h)')
print(f'  RMSE : {rmse_lstm:.2f} h   (Baseline: {rmse_bl:.2f} h)')
print(f'  MAPE : {mape_lstm:.1f}%')
logger.info(f'Hours regression | MAE={mae_lstm:.2f} RMSE={rmse_lstm:.2f} MAPE={mape_lstm:.1f}%')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ah, ph, alpha=0.3, s=10, label='predictions')
mn, mx = min(ah.min(), ph.min()), max(ah.max(), ph.max())
ax.plot([mn, mx], [mn, mx], 'r--', label='perfect')
ax.set_xlabel('Actual hours to next stage')
ax.set_ylabel('Predicted hours to next stage')
ax.set_title('Hours-to-Next: Actual vs Predicted')
ax.legend()
plt.tight_layout()
p = save_fig(fig, 'hours_actual_vs_predicted')
print(f'Scatter plot saved → {p}')
plt.show()
plt.close(fig)

# ── 13.3  Transition binary metrics ──────────────────────────
for label, true_arr, prob_arr, pred_arr, key in [
    ('24h Transition', y_t24_te, pred_t24_prob, pred_t24_binary, 'trans_24h'),
    ('48h Transition', y_t48_te, pred_t48_prob, pred_t48_binary, 'trans_48h'),
]:
    valid = ~np.isnan(true_arr)
    t     = true_arr[valid].astype(int)
    pb    = prob_arr[valid]
    pbd   = pred_arr[valid]
    acc   = accuracy_score(t, pbd)
    prec  = precision_score(t, pbd, zero_division=0)
    rec   = recall_score(t, pbd, zero_division=0)
    f1    = f1_score(t, pbd, zero_division=0)
    auc   = roc_auc_score(t, pb) if len(np.unique(t)) > 1 else float('nan')

    print(f'\n── {label} ──')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1        : {f1:.4f}')
    print(f'  ROC-AUC   : {auc:.4f}')

    lstm_metrics[f'{key}_accuracy']  = float(acc)
    lstm_metrics[f'{key}_precision'] = float(prec)
    lstm_metrics[f'{key}_recall']    = float(rec)
    lstm_metrics[f'{key}_f1']        = float(f1)
    lstm_metrics[f'{key}_roc_auc']   = float(auc)

    cm2 = confusion_matrix(t, pbd)
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(cm2).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'CM — {label}')
    plt.tight_layout()
    p = save_fig(fig, f'confusion_matrix_{key}')
    print(f'CM saved → {p}')
    plt.show()
    plt.close(fig)

# ── Save all LSTM metrics ─────────────────────────────────────
with open(os.path.join(METRICS_DIR, 'lstm_metrics.json'), 'w') as f:
    json.dump(lstm_metrics, f, indent=2)
print(f'\n✅ LSTM evaluation metrics saved.')
logger.info(f'Evaluation done | metrics={lstm_metrics}')



── CURRENT STAGE ──
                      precision    recall  f1-score   support

            seedling       1.00      0.91      0.96       385
    early_vegetative       0.99      0.97      0.98       624
flowering_initiation       0.85      0.65      0.73       288
           flowering       0.76      0.94      0.84       456
              unripe       0.99      0.98      0.99       792
                ripe       0.93      1.00      0.96       240

            accuracy                           0.93      2785
           macro avg       0.92      0.91      0.91      2785
        weighted avg       0.93      0.93      0.93      2785

Confusion matrix saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\confusion_matrix_current_stage.png

── NEXT STAGE (valid rows only) ──
                      precision    recall  f1-score   support

            seedling       0.00      0.00      0.00         0
    early_vegetative       1.00      0.93     

2026-03-10 19:42:10,389 | INFO | Hours regression | MAE=109.51 RMSE=137.05 MAPE=174.5%


Confusion matrix saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\confusion_matrix_next_stage.png

── Hours-to-next regression ──
  MAE  : 109.51 h   (Baseline: 1.56 h)
  RMSE : 137.05 h   (Baseline: 7.73 h)
  MAPE : 174.5%
Scatter plot saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\hours_actual_vs_predicted.png

── 24h Transition ──
  Accuracy  : 0.9395
  Precision : 0.3731
  Recall    : 0.4167
  F1        : 0.3937
  ROC-AUC   : 0.8116
CM saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\confusion_matrix_trans_24h.png

── 48h Transition ──
  Accuracy  : 0.8578
  Precision : 0.3460
  Recall    : 0.5708
  F1        : 0.4308
  ROC-AUC   : 0.8106


2026-03-10 19:42:11,788 | INFO | Evaluation done | metrics={'current_stage_accuracy': 0.9289048473967684, 'next_stage_accuracy': 0.9489194499017681, 'hours_mae': 109.50504302978516, 'hours_rmse': 137.04521379822063, 'hours_mape_pct': 174.45542907714844, 'trans_24h_accuracy': 0.9394891944990177, 'trans_24h_precision': 0.373134328358209, 'trans_24h_recall': 0.4166666666666667, 'trans_24h_f1': 0.3937007874015748, 'trans_24h_roc_auc': 0.8116426116838489, 'trans_48h_accuracy': 0.8577603143418467, 'trans_48h_precision': 0.34595959595959597, 'trans_48h_recall': 0.5708333333333333, 'trans_48h_f1': 0.4308176100628931, 'trans_48h_roc_auc': 0.8105911062906724}


CM saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\confusion_matrix_trans_48h.png

✅ LSTM evaluation metrics saved.


## SECTION 14 — Real Scenario Interpretation

In [17]:
# ─────────────────────────────────────────────────────────────
# SECTION 14: Realistic Inference Examples
# ─────────────────────────────────────────────────────────────

N_EXAMPLES = 8
example_indices = np.random.choice(len(X_te_sc), N_EXAMPLES, replace=False)

print('=' * 80)
print('TOMATO PROGRESSION — REAL SCENARIO INFERENCE EXAMPLES')
print('=' * 80)

sample_rows = []
for idx in example_indices:
    pc     = pred_cur_labels[idx]
    pn     = pred_nxt_labels[idx]
    ph_val = pred_hrs_raw[idx]
    pt24   = pred_t24_prob[idx]
    pt48   = pred_t48_prob[idx]

    ac     = y_cur_te[idx]
    an     = y_nxt_te[idx]
    ah_val = y_hrs_te[idx]
    at24   = y_t24_te[idx]
    at48   = y_t48_te[idx]

    print(f'\nSample #{idx}')
    print(f'  Predicted current stage  : {INT_TO_STAGE[pc]:25s} | Actual: {INT_TO_STAGE[ac]}')
    if not np.isnan(an):
        print(f'  Predicted next stage     : {INT_TO_STAGE[pn]:25s} | Actual: {INT_TO_STAGE[int(an)]}')
        print(f'  Predicted hrs to next    : {ph_val:8.1f} h               | Actual: {ah_val:.1f} h')
        print(f'  Prob transition in 24h   : {pt24:8.3f}                | Actual: {int(at24)}')
        print(f'  Prob transition in 48h   : {pt48:8.3f}                | Actual: {int(at48)}')
    else:
        print('  [Row is at final RIPE stage — no next transition target]')
    print('-' * 60)

    sample_rows.append({
        'sample_idx'             : int(idx),
        'pred_current_stage'     : INT_TO_STAGE[pc],
        'actual_current_stage'   : INT_TO_STAGE[ac],
        'pred_next_stage'        : INT_TO_STAGE[pn],
        'actual_next_stage'      : INT_TO_STAGE[int(an)] if not np.isnan(an) else 'N/A',
        'pred_hours_to_next'     : round(float(ph_val), 2),
        'actual_hours_to_next'   : round(float(ah_val), 2) if not np.isnan(ah_val) else None,
        'pred_trans_prob_24h'    : round(float(pt24), 4),
        'actual_trans_24h'       : int(at24) if not np.isnan(at24) else None,
        'pred_trans_prob_48h'    : round(float(pt48), 4),
        'actual_trans_48h'       : int(at48) if not np.isnan(at48) else None,
    })

# ── Save prediction samples to CSV ────────────────────────────
samples_df = pd.DataFrame(sample_rows)
samples_path = os.path.join(REPORTS_DIR, 'prediction_samples.csv')
samples_df.to_csv(samples_path, index=False)
print(f'\nPrediction samples saved → {samples_path}')
logger.info(f'Prediction samples ({N_EXAMPLES} examples) saved.')


2026-03-10 19:42:40,996 | INFO | Prediction samples (8 examples) saved.


TOMATO PROGRESSION — REAL SCENARIO INFERENCE EXAMPLES

Sample #2079
  Predicted current stage  : unripe                    | Actual: unripe
  Predicted next stage     : ripe                      | Actual: ripe
  Predicted hrs to next    :    402.5 h               | Actual: 466.0 h
  Prob transition in 24h   :    0.000                | Actual: 0
  Prob transition in 48h   :    0.000                | Actual: 0
------------------------------------------------------------

Sample #2771
  Predicted current stage  : ripe                      | Actual: ripe
  [Row is at final RIPE stage — no next transition target]
------------------------------------------------------------

Sample #1465
  Predicted current stage  : flowering                 | Actual: flowering
  Predicted next stage     : unripe                    | Actual: unripe
  Predicted hrs to next    :    197.7 h               | Actual: 288.0 h
  Prob transition in 24h   :    0.005                | Actual: 0
  Prob transition in 48h 

## SECTION 15 — Visualization

In [18]:
# ─────────────────────────────────────────────────────────────
# SECTION 15: Visualizations
# ─────────────────────────────────────────────────────────────

# ── 15.1  Stage distribution ──────────────────────────────────
stage_counts = df['stage'].value_counts().reindex(STAGE_ORDER, fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4))
stage_counts.plot(kind='bar', ax=ax, color=sns.color_palette('muted', N_STAGES))
ax.set_title('Stage Distribution Across All Cycles')
ax.set_xlabel('Stage')
ax.set_ylabel('Row Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
p = save_fig(fig, 'stage_distribution')
print(f'Stage distribution saved → {p}')
plt.show()
plt.close(fig)

# ── 15.2  Cycle duration distribution ─────────────────────────
cycle_durations = cycle_summary['duration_hours'].dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cycle_durations, bins=20, edgecolor='white', color='steelblue')
ax.set_title('Cycle Duration Distribution')
ax.set_xlabel('Duration (hours)')
ax.set_ylabel('Number of Cycles')
plt.tight_layout()
p = save_fig(fig, 'cycle_duration_distribution')
print(f'Cycle duration histogram saved → {p}')
plt.show()
plt.close(fig)

# ── 15.3  Progression timelines for 3 sample cycles ──────────
sample_cycles = df['cycle_id'].unique()[:3]
palette   = sns.color_palette('tab10', N_STAGES)
color_map = {s: palette[i] for i, s in enumerate(STAGE_ORDER)}

fig, axes = plt.subplots(len(sample_cycles), 1, figsize=(14, 3 * len(sample_cycles)))
if len(sample_cycles) == 1:
    axes = [axes]

for ax, cid in zip(axes, sample_cycles):
    cdf = df[df['cycle_id'] == cid].sort_values('timestamp')
    for _, row in cdf.iterrows():
        ax.axvline(row['timestamp'], color=color_map[row['stage']], alpha=0.4, linewidth=1.2)
    ax.set_title(f'Cycle {cid} — Stage Timeline')
    ax.set_xlabel('Time')
    ax.set_yticks([])
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=color_map[s], label=s)
        for s in STAGE_ORDER if s in cdf['stage'].values
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
plt.suptitle('Stage Progression Timelines')
plt.tight_layout()
p = save_fig(fig, 'stage_progression_timelines')
print(f'Stage timelines saved → {p}')
plt.show()
plt.close(fig)

# ── 15.4  Transition probability histograms ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, probs, true_vals) in zip(axes, [
    ('24h Transition Probabilities', pred_t24_prob, y_t24_te),
    ('48h Transition Probabilities', pred_t48_prob, y_t48_te),
]):
    valid = ~np.isnan(true_vals)
    for lbl, color in [(0, 'steelblue'), (1, 'tomato')]:
        mask = (true_vals[valid] == lbl)
        ax.hist(probs[valid][mask], bins=30, alpha=0.6,
                color=color, label=f'Actual={lbl}', density=True)
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel('Predicted Probability')
    ax.legend()
plt.tight_layout()
p = save_fig(fig, 'transition_probability_histograms')
print(f'Transition histograms saved → {p}')
plt.show()
plt.close(fig)

# ── 15.5  Feature importance (RF baseline) ───────────────────
top_n = 20
top_feats = feat_imp.nlargest(top_n)
fig, ax = plt.subplots(figsize=(9, 6))
top_feats.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Top-{top_n} RF Feature Importances (Current Stage)')
ax.set_xlabel('Importance')
plt.tight_layout()
p = save_fig(fig, 'rf_feature_importances')
print(f'Feature importances saved → {p}')
plt.show()
plt.close(fig)

print('\n✅ All visualizations saved.')
logger.info('All visualizations saved to plots directory.')


Stage distribution saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\stage_distribution.png
Cycle duration histogram saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\cycle_duration_distribution.png
Stage timelines saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\stage_progression_timelines.png
Transition histograms saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\transition_probability_histograms.png


2026-03-10 19:43:42,557 | INFO | All visualizations saved to plots directory.


Feature importances saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\rf_feature_importances.png

✅ All visualizations saved.


## SECTION 16 — Model Saving & Artifact Export

In [19]:
# ─────────────────────────────────────────────────────────────
# SECTION 16: Save Artifacts
# ─────────────────────────────────────────────────────────────

# ── 16.1  Save Keras model ────────────────────────────────────
model.save(MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')
logger.info(f'Model saved → {MODEL_PATH}')

# ── 16.2  Save feature scaler ─────────────────────────────────
SCALER_PATH = os.path.join(ARTIFACTS_DIR, 'feature_scaler.pkl')
with open(SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Scaler saved → {SCALER_PATH}')

# ── 16.3  Save full inference config ─────────────────────────
CONFIG = {
    'run_id'        : RUN_ID,
    'model_path'    : MODEL_PATH,
    'scaler_path'   : SCALER_PATH,
    'seq_len'       : SEQ_LEN,
    'n_features'    : n_features,
    'n_stages'      : N_STAGES,
    'stage_to_int'  : STAGE_TO_INT,
    'int_to_stage'  : {str(k): v for k, v in INT_TO_STAGE.items()},
    'feature_cols'  : FEATURE_COLS,
    'hrs_mean'      : float(HRS_MEAN),
    'hrs_std'       : float(HRS_STD),
    'stage_order'   : STAGE_ORDER,
    'rolling_windows': ROLLING_WINDOWS,
    'lag_steps'     : LAG_STEPS,
    'random_seed'   : RANDOM_SEED,
}
CONFIG_PATH = os.path.join(ARTIFACTS_DIR, 'inference_config.json')
with open(CONFIG_PATH, 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(f'Inference config saved → {CONFIG_PATH}')

# ── 16.4  Save combined per-stage metrics (train distribution) ─
per_stage_dist = {}
for split, y_arr in [('train', y_cur_tr), ('val', y_cur_va), ('test', y_cur_te)]:
    per_stage_dist[split] = {
        STAGE_ORDER[i]: int((y_arr == i).sum()) for i in range(N_STAGES)
    }
with open(os.path.join(METRICS_DIR, 'per_stage_distribution.json'), 'w') as f:
    json.dump(per_stage_dist, f, indent=2)

# ── 16.5  Save full test predictions CSV ─────────────────────
te_preds_df = pd.DataFrame({
    'pred_current_stage'  : [INT_TO_STAGE[l] for l in pred_cur_labels],
    'actual_current_stage': [INT_TO_STAGE[l] for l in y_cur_te],
    'pred_next_stage'     : [INT_TO_STAGE[l] for l in pred_nxt_labels],
    'actual_next_stage'   : [INT_TO_STAGE[int(l)] if not np.isnan(l) else 'N/A' for l in y_nxt_te],
    'pred_hours_to_next'  : pred_hrs_raw.round(2),
    'actual_hours_to_next': y_hrs_te.round(2),
    'pred_trans_prob_24h' : pred_t24_prob.round(4),
    'actual_trans_24h'    : y_t24_te,
    'pred_trans_prob_48h' : pred_t48_prob.round(4),
    'actual_trans_48h'    : y_t48_te,
})
te_preds_path = os.path.join(REPORTS_DIR, 'test_predictions_full.csv')
te_preds_df.to_csv(te_preds_path, index=False)
print(f'Full test predictions saved → {te_preds_path}')

print(f'\n✅ All artifacts saved to:\n  {ARTIFACTS_DIR}')
logger.info('All artifacts saved.')


2026-03-10 19:43:55,295 | INFO | Model saved → E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_progression_20260310_192038.keras
2026-03-10 19:43:55,408 | INFO | All artifacts saved.


Model saved → E:\AgriTwin-GH\src\agritwin_gh\models\growth_stage_progression_20260310_192038.keras
Scaler saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\feature_scaler.pkl
Inference config saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\inference_config.json
Full test predictions saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\test_predictions_full.csv

✅ All artifacts saved to:
  E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038


## SECTION 17 — Inference Utilities

In [21]:
# ─────────────────────────────────────────────────────────────
# SECTION 17: Inference Utilities
# ─────────────────────────────────────────────────────────────

def decode_stage(idx):
    """Convert integer index to stage label."""
    return INT_TO_STAGE.get(int(np.round(idx)), 'Unknown')


def preprocess_new_data(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Standardise column names, fill gaps, and run feature engineering
    on a raw sensor DataFrame using the same pipeline as training.
    Returns a DataFrame with FEATURE_COLS ready for sequence building.
    """
    df_p = standardize_columns(df_raw.copy())
    detect_required_columns(df_p)
    df_p = normalise_stage_labels(df_p)

    # build targets so that engineer_features_for_cycle can run smoothly
    df_p = build_progression_targets(df_p)

    df_p['hour_of_day'] = pd.to_datetime(df_p['timestamp']).dt.hour

    # Engineer features (use a fake cycle_id so the function works)
    df_p['cycle_id'] = df_p.get('cycle_id', 'inference')
    cycle_dfs = []
    for cid, cdf in df_p.groupby('cycle_id'):
        cycle_dfs.append(engineer_features_for_cycle(cdf))
    df_feat_p = pd.concat(cycle_dfs).reset_index(drop=True)

    # Fill NaNs that arise from rolling ops at the start of the series
    df_feat_p[FEATURE_COLS] = df_feat_p[FEATURE_COLS].bfill().fillna(0)
    return df_feat_p


def build_recent_sequence(df_feats: pd.DataFrame,
                           seq_len: int = SEQ_LEN) -> np.ndarray:
    """
    Take the *last* `seq_len` rows of `df_feats` and return a scaled
    (1, seq_len, n_features) array suitable for model.predict().
    Raises ValueError if fewer than `seq_len` rows are available.
    """
    if len(df_feats) < seq_len:
        raise ValueError(
            f'Need ≥ {seq_len} rows; got {len(df_feats)}.')
    raw = df_feats[FEATURE_COLS].values[-seq_len:]          # (seq_len, F)
    scaled = scaler.transform(raw)                           # (seq_len, F)
    return scaled[np.newaxis, ...]                           # (1, seq_len, F)


def predict_progression(sequence_3d: np.ndarray) -> dict:
    """
    Run a single (1, seq_len, n_features) array through the LSTM model
    and return a human-readable prediction dictionary.

    Keys returned:
      current_stage, next_stage,
      hours_to_transition, transition_prob_24h, transition_prob_48h
    """
    preds = model.predict(sequence_3d, verbose=0)
    cur_logits, nxt_logits, hrs_pred, t24_logit, t48_logit = preds

    cur_idx   = int(np.argmax(cur_logits[0]))
    nxt_idx   = int(np.argmax(nxt_logits[0]))
    hrs_raw   = float(hrs_pred[0, 0])
    hrs_actual = hrs_raw * HRS_STD + HRS_MEAN

    t24_prob  = float(tf.sigmoid(t24_logit[0, 0]).numpy())
    t48_prob  = float(tf.sigmoid(t48_logit[0, 0]).numpy())

    return {
        'current_stage'         : decode_stage(cur_idx),
        'next_stage'            : decode_stage(nxt_idx),
        'hours_to_transition'   : round(max(0.0, hrs_actual), 2),
        'transition_prob_24h'   : round(t24_prob, 4),
        'transition_prob_48h'   : round(t48_prob, 4),
        'will_transition_24h'   : t24_prob >= 0.5,
        'will_transition_48h'   : t48_prob >= 0.5,
    }


# ── Demo: run predict_progression on the first test sequence ──
demo_seq  = X_te_sc[0:1]   # already scaled (1, SEQ_LEN, n_features)
demo_pred = predict_progression(demo_seq)
demo_true_cur = decode_stage(y_cur_te[0])
demo_true_nxt = decode_stage(y_nxt_te[0]) if not np.isnan(y_nxt_te[0]) else 'N/A'

print('── Inference Demo (Test Sample #0) ──')
print(f"  True current stage :  {demo_true_cur}")
print(f"  Pred current stage :  {demo_pred['current_stage']}")
print(f"  True next stage    :  {demo_true_nxt}")
print(f"  Pred next stage    :  {demo_pred['next_stage']}")
print(f"  Pred hours to next :  {demo_pred['hours_to_transition']} h")
print(f"  Trans prob  24 h   :  {demo_pred['transition_prob_24h']:.2%}")
print(f"  Trans prob  48 h   :  {demo_pred['transition_prob_48h']:.2%}")
print(f"  Will transit 24 h? :  {demo_pred['will_transition_24h']}")
print(f"  Will transit 48 h? :  {demo_pred['will_transition_48h']}")

# Save demo result
DEMO_PATH = os.path.join(REPORTS_DIR, 'inference_demo.json')
_np_default = lambda o: float(o) if isinstance(o, np.floating) else bool(o) if isinstance(o, np.bool_) else int(o) if isinstance(o, np.integer) else str(o)
with open(DEMO_PATH, 'w') as f:
    json.dump({
        'true_current_stage': demo_true_cur,
        'true_next_stage'   : demo_true_nxt,
        **demo_pred,
    }, f, indent=2, default=_np_default)
print(f'\nDemo result saved → {DEMO_PATH}')
logger.info('Section 17 complete — inference utilities defined and demo run.')
print('✅ Section 17 complete.')


2026-03-10 19:44:47,995 | INFO | Section 17 complete — inference utilities defined and demo run.


── Inference Demo (Test Sample #0) ──
  True current stage :  seedling
  Pred current stage :  seedling
  True next stage    :  early_vegetative
  Pred next stage    :  early_vegetative
  Pred hours to next :  172.0500030517578 h
  Trans prob  24 h   :  50.29%
  Trans prob  48 h   :  53.38%
  Will transit 24 h? :  True
  Will transit 48 h? :  True

Demo result saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\inference_demo.json
✅ Section 17 complete.


## SECTION 18 — Final Summary

In [23]:
# ─────────────────────────────────────────────────────────────
# SECTION 18: Final Run Summary
# ─────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score

# ── Re-compute headline metrics ───────────────────────────────
final_cur_acc  = float((pred_cur_labels == y_cur_te).mean())
final_nxt_mask = ~np.isnan(y_nxt_te)
final_nxt_acc  = float(
    (pred_nxt_labels[final_nxt_mask] == y_nxt_te[final_nxt_mask]).mean()
) if final_nxt_mask.any() else float('nan')

final_mae   = float(np.mean(np.abs(pred_hrs_raw  * HRS_STD + HRS_MEAN
                                    - y_hrs_te)))
final_rmse  = float(np.sqrt(np.mean((pred_hrs_raw * HRS_STD + HRS_MEAN
                                      - y_hrs_te) ** 2)))
final_mape  = float(np.mean(
    np.abs(pred_hrs_raw * HRS_STD + HRS_MEAN - y_hrs_te)
    / np.clip(np.abs(y_hrs_te), 1e-6, None)
) * 100)

final_t24_mask = ~np.isnan(y_t24_te)
final_t48_mask = ~np.isnan(y_t48_te)
final_t24_f1 = float(f1_score(y_t24_te[final_t24_mask].astype(int), pred_t24_binary[final_t24_mask], zero_division=0))
final_t48_f1 = float(f1_score(y_t48_te[final_t48_mask].astype(int), pred_t48_binary[final_t48_mask], zero_division=0))

# ── Build summary dict ────────────────────────────────────────
run_summary = {
    'run_id'               : RUN_ID,
    'model_path'           : MODEL_PATH,
    'artifacts_dir'        : ARTIFACTS_DIR,
    'dataset_path'         : DATA_PATH,

    'split': {
        'train_cycles': len(train_cycles),
        'val_cycles'  : len(val_cycles),
        'test_cycles' : len(test_cycles),
        'train_seqs'  : int(X_tr.shape[0]),
        'val_seqs'    : int(X_va.shape[0]),
        'test_seqs'   : int(X_te.shape[0]),
    },

    'baselines': {
        'rf_current_stage_accuracy': float(rf_acc),
        'xgb_hours_mae'            : float(mae_bl),
        'xgb_hours_rmse'           : float(rmse_bl),
    },

    'lstm_metrics': {
        'current_stage_accuracy': round(final_cur_acc,  4),
        'next_stage_accuracy'   : round(final_nxt_acc,  4) if not np.isnan(final_nxt_acc) else None,
        'hours_mae'             : round(final_mae,       4),
        'hours_rmse'            : round(final_rmse,      4),
        'hours_mape_pct'        : round(final_mape,      4),
        'transition_24h_f1'     : round(final_t24_f1,   4),
        'transition_48h_f1'     : round(final_t48_f1,   4),
    },

    'improvement_vs_baseline': {
        'hours_mae_delta'  : round(mae_bl  - final_mae,  4),
        'hours_rmse_delta' : round(rmse_bl - final_rmse, 4),
    },

    'artifacts': sorted([
        str(Path(root) / f)
        for root, _, files in os.walk(ARTIFACTS_DIR)
        for f in files
    ]),
}

# ── Save as JSON ──────────────────────────────────────────────
SUMMARY_JSON = os.path.join(ARTIFACTS_DIR, 'run_summary.json')
with open(SUMMARY_JSON, 'w') as fh:
    json.dump(run_summary, fh, indent=2, default=str)

# ── Save as human-readable TXT ────────────────────────────────
SUMMARY_TXT = os.path.join(ARTIFACTS_DIR, 'run_summary.txt')
sep = '─' * 60
lines = [
    sep,
    f'  TOMATO GROWTH PROGRESSION — RUN SUMMARY',
    f'  Run ID : {RUN_ID}',
    sep,
    '',
    'Dataset',
    f"  Path   : {DATA_PATH}",
    '',
    'Train / Val / Test split (cycles)',
    f"  Train  : {run_summary['split']['train_cycles']}  ({run_summary['split']['train_seqs']:,} seqs)",
    f"  Val    : {run_summary['split']['val_cycles']}  ({run_summary['split']['val_seqs']:,} seqs)",
    f"  Test   : {run_summary['split']['test_cycles']}  ({run_summary['split']['test_seqs']:,} seqs)",
    '',
    'Baseline (RF / XGBoost)',
    f"  RF current-stage acc : {run_summary['baselines']['rf_current_stage_accuracy']:.4f}",
    f"  XGB hours    MAE     : {run_summary['baselines']['xgb_hours_mae']:.4f}",
    f"  XGB hours    RMSE    : {run_summary['baselines']['xgb_hours_rmse']:.4f}",
    '',
    'LSTM Test Metrics',
    f"  Current stage acc    : {run_summary['lstm_metrics']['current_stage_accuracy']:.4f}",
    f"  Next stage acc       : {run_summary['lstm_metrics']['next_stage_accuracy']}",
    f"  Hours  MAE           : {run_summary['lstm_metrics']['hours_mae']:.4f}",
    f"  Hours  RMSE          : {run_summary['lstm_metrics']['hours_rmse']:.4f}",
    f"  Hours  MAPE          : {run_summary['lstm_metrics']['hours_mape_pct']:.2f}%",
    f"  Transition 24h F1    : {run_summary['lstm_metrics']['transition_24h_f1']:.4f}",
    f"  Transition 48h F1    : {run_summary['lstm_metrics']['transition_48h_f1']:.4f}",
    '',
    'Improvement vs Baseline',
    f"  MAE   Δ (positive = better) : {run_summary['improvement_vs_baseline']['hours_mae_delta']:.4f}",
    f"  RMSE  Δ (positive = better) : {run_summary['improvement_vs_baseline']['hours_rmse_delta']:.4f}",
    '',
    f'Artifacts directory : {ARTIFACTS_DIR}',
    sep,
]
with open(SUMMARY_TXT, 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(lines))

# ── Print to notebook output ──────────────────────────────────
print('\n'.join(lines))

logger.info('Section 18 complete — run summary saved.')
logger.info(f'Run summary JSON → {SUMMARY_JSON}')
logger.info(f'Run summary TXT  → {SUMMARY_TXT}')
print(f'\n✅ Notebook complete.  All outputs under:\n  {ARTIFACTS_DIR}')


2026-03-10 19:45:35,901 | INFO | Section 18 complete — run summary saved.


2026-03-10 19:45:35,904 | INFO | Run summary JSON → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\run_summary.json
2026-03-10 19:45:35,907 | INFO | Run summary TXT  → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038\run_summary.txt


────────────────────────────────────────────────────────────
  TOMATO GROWTH PROGRESSION — RUN SUMMARY
  Run ID : 20260310_192038
────────────────────────────────────────────────────────────

Dataset
  Path   : E:\AgriTwin-GH\data\processed\Growth Progression\tomato_growth_progression_synthetic_hourly.csv

Train / Val / Test split (cycles)
  Train  : 2  (5,282 seqs)
  Val    : 1  (2,689 seqs)
  Test   : 1  (2,785 seqs)

Baseline (RF / XGBoost)
  RF current-stage acc : 1.0000
  XGB hours    MAE     : 1.5575
  XGB hours    RMSE    : 7.7259

LSTM Test Metrics
  Current stage acc    : 0.9289
  Next stage acc       : 0.9489
  Hours  MAE           : nan
  Hours  RMSE          : nan
  Hours  MAPE          : nan%
  Transition 24h F1    : 0.3937
  Transition 48h F1    : 0.4308

Improvement vs Baseline
  MAE   Δ (positive = better) : nan
  RMSE  Δ (positive = better) : nan

Artifacts directory : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\growth_stage_progression_20260310_192038
────────────